# 02 — Walk-Forward Validation

**Objetivo:** Validar o modelo XGBoost através de múltiplos folds temporais, cobrindo sazonalidade e tendência de preços.

**Por que não 80/20 aleatório?**  
Um split aleatório vaza informação temporal: o modelo pode aprender padrões de imóveis "futuros" durante o treino, inflando artificialmente as métricas. O walk-forward garante que cada fold simula o cenário real de produção — treinar no passado, avaliar no futuro.

```
Fold 1: Treino [2014-05 → 2014-10] | Teste [2014-10 → 2015-01]
Fold 2: Treino [2014-05 → 2015-01] | Teste [2015-01 → 2015-04]
Fold 3: Treino [2014-05 → 2015-04] | Teste [2015-04 → 2015-08]
```

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

from app.ml.preprocess import (
    load_raw_kc_house_sales,
    load_zipcode_demographics_if_present,
    merge_housing_with_zipcode_demographics,
    make_xgboost_preprocessor,
    make_ridge_baseline_preprocessor,
)
from app.ml.feature_engineering import DerivedHousingFeatures, TARGET

pio.templates.default = 'plotly_white'

BLUE        = '#2563EB'
BLUE_DARK   = '#1D4ED8'
BLUE_LIGHT  = '#93C5FD'
RED         = '#DC2626'
RED_DARK    = '#991B1B'
RED_LIGHT   = '#FCA5A5'
GREEN       = '#16A34A'
GRAY        = '#6B7280'
ORANGE      = '#EA580C'

FOLD_COLORS = {
    'train': [BLUE_LIGHT, BLUE, BLUE_DARK],
    'test':  [RED_LIGHT,  RED,  RED_DARK],
}

print('Libs ok.')

## 1. Carga de dados

In [ ]:
houses = load_raw_kc_house_sales(Path('../data/raw/kc_house_data.csv'))
demographics = load_zipcode_demographics_if_present(Path('../data/raw/zipcode_demographics.csv'))
df = merge_housing_with_zipcode_demographics(houses, demographics)

df['date_dt'] = pd.to_datetime(df['date'].astype(str).str[:8], format='%Y%m%d')
df = df.sort_values('date_dt').reset_index(drop=True)

print(f'Total     : {len(df):,} registros')
print(f'Período   : {df["date_dt"].min().date()} → {df["date_dt"].max().date()}')
df[['date_dt', 'price']].head(3)

## 2. Definição dos folds — Expanding Window

In [ ]:
FOLDS = [
    {
        'name': 'Fold 1',
        'train_end':   pd.Timestamp('2014-10-01'),
        'test_start':  pd.Timestamp('2014-10-01'),
        'test_end':    pd.Timestamp('2015-01-01'),
    },
    {
        'name': 'Fold 2',
        'train_end':   pd.Timestamp('2015-01-01'),
        'test_start':  pd.Timestamp('2015-01-01'),
        'test_end':    pd.Timestamp('2015-04-01'),
    },
    {
        'name': 'Fold 3',
        'train_end':   pd.Timestamp('2015-04-01'),
        'test_start':  pd.Timestamp('2015-04-01'),
        'test_end':    pd.Timestamp('2015-08-01'),
    },
]

TRAIN_START = df['date_dt'].min()

print('Folds definidos:')
for f in FOLDS:
    n_train = ((df['date_dt'] >= TRAIN_START) & (df['date_dt'] < f['train_end'])).sum()
    n_test  = ((df['date_dt'] >= f['test_start']) & (df['date_dt'] < f['test_end'])).sum()
    print(f"  {f['name']} | treino: {n_train:,} | teste: {n_test:,}")

In [ ]:
# Diagrama de Gantt dos folds
gantt_rows = []
for i, fold in enumerate(FOLDS):
    gantt_rows.append(dict(
        Fold=fold['name'], Tipo='Treino',
        Start=TRAIN_START, Finish=fold['train_end'],
        Color=FOLD_COLORS['train'][i],
    ))
    gantt_rows.append(dict(
        Fold=fold['name'], Tipo='Teste',
        Start=fold['test_start'], Finish=fold['test_end'],
        Color=FOLD_COLORS['test'][i],
    ))

df_gantt = pd.DataFrame(gantt_rows)

fig = px.timeline(
    df_gantt,
    x_start='Start', x_end='Finish',
    y='Fold', color='Tipo',
    color_discrete_map={'Treino': BLUE, 'Teste': RED},
    title='Walk-Forward Validation — Expanding Window',
    labels={'Tipo': ''},
    height=280,
    opacity=0.85,
)
fig.update_yaxes(autorange='reversed')
fig.update_layout(
    title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
    margin=dict(l=80, r=20, t=60, b=40),
    xaxis_title='',
)
fig.show()

## 3. Funções auxiliares

In [ ]:
XGB_PARAMS = {
    'n_estimators': 400,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'tree_method': 'hist',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
}


def compute_metrics(y_true_log, y_pred_log, label=''):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return {
        'label': label,
        'RMSE':  float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':   float(mean_absolute_error(y_true, y_pred)),
        'R2':    float(r2_score(y_true, y_pred)),
        'MAPE':  float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100),
        'n':     len(y_true),
    }


def split_fold(df, fold):
    train_mask = (df['date_dt'] >= TRAIN_START) & (df['date_dt'] < fold['train_end'])
    test_mask  = (df['date_dt'] >= fold['test_start']) & (df['date_dt'] < fold['test_end'])
    X_tr = df[train_mask].drop(columns=[TARGET])
    y_tr = df[train_mask][TARGET]
    X_te = df[test_mask].drop(columns=[TARGET])
    y_te = df[test_mask][TARGET]
    return X_tr, X_te, y_tr, y_te


print('Funções definidas.')

## 4. Treinamento e avaliação por fold

In [ ]:
results_xgb   = []
results_ridge = []
fold_details  = []

for fold in FOLDS:
    X_tr, X_te, y_tr, y_te = split_fold(df, fold)
    y_tr_log = np.log1p(y_tr.values)
    y_te_log = np.log1p(y_te.values)

    print(f"\n{'='*55}")
    print(f"{fold['name']} | treino: {len(X_tr):,} | teste: {len(X_te):,}")

    # Ridge
    ridge_pipe = Pipeline([
        ('features', DerivedHousingFeatures()),
        ('prep', make_ridge_baseline_preprocessor()),
        ('model', Ridge(alpha=10.0)),
    ])
    ridge_pipe.fit(X_tr, y_tr_log)
    r_pred = ridge_pipe.predict(X_te)
    r_m = compute_metrics(y_te_log, r_pred, label=fold['name'])
    results_ridge.append(r_m)
    print(f"  Ridge   — R²: {r_m['R2']:.4f} | MAE: ${r_m['MAE']:,.0f} | MAPE: {r_m['MAPE']:.2f}%")

    # XGBoost
    xgb_pipe = Pipeline([
        ('features', DerivedHousingFeatures()),
        ('prep', make_xgboost_preprocessor()),
        ('model', XGBRegressor(**XGB_PARAMS)),
    ])
    xgb_pipe.fit(X_tr, y_tr_log)
    x_pred = xgb_pipe.predict(X_te)
    x_m = compute_metrics(y_te_log, x_pred, label=fold['name'])
    results_xgb.append(x_m)
    print(f"  XGBoost — R²: {x_m['R2']:.4f} | MAE: ${x_m['MAE']:,.0f} | MAPE: {x_m['MAPE']:.2f}%")

    fold_details.append({
        'fold': fold['name'],
        'n_train': len(X_tr),
        'n_test': len(X_te),
        'xgb_metrics': x_m,
        'ridge_metrics': r_m,
        'y_te_log': y_te_log,
        'xgb_pred_log': x_pred,
        'ridge_pred_log': r_pred,
        'X_te': X_te,
    })

df_xgb   = pd.DataFrame(results_xgb).set_index('label')
df_ridge = pd.DataFrame(results_ridge).set_index('label')
print(f"\n{'='*55}")
print('Treinamento completo.')

## 5. Comparação de métricas entre folds

In [ ]:
metrics_plot = ['R2', 'MAE', 'MAPE']
titles = ['R² (maior = melhor)', 'MAE — erro absoluto médio (k$)', 'MAPE — erro percentual médio (%)']

fig = make_subplots(rows=1, cols=3, subplot_titles=titles)

fold_names = [f['name'] for f in FOLDS]

for col_idx, metric in enumerate(metrics_plot, start=1):
    xgb_vals   = [r[metric] for r in results_xgb]
    ridge_vals = [r[metric] for r in results_ridge]

    if metric == 'MAE':
        xgb_vals   = [v / 1_000 for v in xgb_vals]
        ridge_vals = [v / 1_000 for v in ridge_vals]

    fig.add_trace(
        go.Bar(name='XGBoost', x=fold_names, y=xgb_vals,
               marker_color=BLUE, showlegend=(col_idx == 1),
               text=[f'{v:.3f}' if metric == 'R2' else f'{v:.1f}' for v in xgb_vals],
               textposition='outside'),
        row=1, col=col_idx
    )
    fig.add_trace(
        go.Bar(name='Ridge', x=fold_names, y=ridge_vals,
               marker_color=RED, showlegend=(col_idx == 1),
               text=[f'{v:.3f}' if metric == 'R2' else f'{v:.1f}' for v in ridge_vals],
               textposition='outside'),
        row=1, col=col_idx
    )

fig.update_layout(
    title_text='Walk-Forward: XGBoost vs Ridge — métricas por fold',
    title_font_size=16,
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
    height=440,
)
fig.show()

## 6. Real vs. Previsto — por fold

In [ ]:
fig = make_subplots(
    rows=1, cols=len(FOLDS),
    subplot_titles=[
        f"{fd['fold']}<br><sup>R²={fd['xgb_metrics']['R2']:.3f} | MAE=${fd['xgb_metrics']['MAE']/1e3:.0f}k</sup>"
        for fd in fold_details
    ],
    shared_yaxes=True,
)

for col_idx, fd in enumerate(fold_details, start=1):
    y_true = np.expm1(fd['y_te_log']) / 1_000
    y_pred = np.expm1(fd['xgb_pred_log']) / 1_000
    lim = max(y_true.max(), y_pred.max()) * 1.05

    fig.add_trace(
        go.Scattergl(
            x=y_true, y=y_pred,
            mode='markers',
            marker=dict(color=BLUE, opacity=0.3, size=4),
            name=fd['fold'],
            showlegend=False,
            hovertemplate='Real: $%{x:.0f}k<br>Previsto: $%{y:.0f}k<extra></extra>',
        ),
        row=1, col=col_idx
    )
    # Linha de perfeição
    fig.add_trace(
        go.Scatter(x=[0, lim], y=[0, lim],
                   mode='lines', line=dict(color=RED, dash='dash', width=1.5),
                   showlegend=(col_idx == 1), name='Perfeição'),
        row=1, col=col_idx
    )
    fig.update_xaxes(title_text='Real (k$)', range=[0, lim], row=1, col=col_idx)

fig.update_yaxes(title_text='Previsto (k$)', row=1, col=1)
fig.update_layout(
    title_text='XGBoost — Real vs. Previsto (walk-forward)',
    title_font_size=16,
    height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='right', x=1),
)
fig.show()

## 7. Resíduos ao longo do tempo

In [ ]:
fig = go.Figure()

fold_plot_colors = [BLUE_LIGHT, BLUE, BLUE_DARK]

for color, fd in zip(fold_plot_colors, fold_details):
    y_true = np.expm1(fd['y_te_log'])
    y_pred = np.expm1(fd['xgb_pred_log'])
    pct_err = (y_pred - y_true) / y_true * 100
    dates   = pd.to_datetime(fd['X_te']['date_dt'].values)

    fig.add_trace(go.Scattergl(
        x=dates, y=pct_err,
        mode='markers',
        marker=dict(color=color, opacity=0.4, size=4),
        name=fd['fold'],
        hovertemplate='%{x|%Y-%m-%d}<br>Erro: %{y:.1f}%<extra></extra>',
    ))

fig.add_hline(y=0,   line_color=RED,  line_width=1.5, line_dash='dash')
fig.add_hline(y=20,  line_color=GRAY, line_width=0.8, line_dash='dot')
fig.add_hline(y=-20, line_color=GRAY, line_width=0.8, line_dash='dot')

fig.update_layout(
    title='Erro percentual XGBoost ao longo do tempo',
    title_font_size=16,
    xaxis_title='Data de venda',
    yaxis_title='Erro percentual (%)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=430,
)
fig.show()

## 8. Sazonalidade do erro — MAPE por mês

In [ ]:
all_dates, all_true, all_pred = [], [], []
for fd in fold_details:
    all_dates.extend(pd.to_datetime(fd['X_te']['date_dt'].values))
    all_true.extend(np.expm1(fd['y_te_log']))
    all_pred.extend(np.expm1(fd['xgb_pred_log']))

df_err = pd.DataFrame({'date': all_dates, 'y_true': all_true, 'y_pred': all_pred})
df_err['abs_pct_err'] = np.abs((df_err['y_pred'] - df_err['y_true']) / df_err['y_true']) * 100
df_err['month'] = df_err['date'].dt.month

month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
mape_month  = df_err.groupby('month')['abs_pct_err'].mean().reset_index()
mape_month['mes'] = mape_month['month'].apply(lambda m: month_names[m-1])

fig = px.bar(
    mape_month, x='mes', y='abs_pct_err',
    color='abs_pct_err',
    color_continuous_scale='RdYlGn_r',
    text_auto='.1f',
    title='Sazonalidade do erro — MAPE médio por mês (todos os folds)',
    labels={'abs_pct_err': 'MAPE (%)', 'mes': 'Mês'},
    height=400,
)
fig.update_traces(texttemplate='%{text}%', textposition='outside', marker_line_width=0)
fig.update_layout(
    title_font_size=16,
    coloraxis_showscale=False,
    yaxis_ticksuffix='%',
)
fig.show()

## 9. Distribuição de erros por fold — Box plot

In [ ]:
rows_box = []
for fd in fold_details:
    y_true = np.expm1(fd['y_te_log'])
    y_pred_xgb   = np.expm1(fd['xgb_pred_log'])
    y_pred_ridge = np.expm1(fd['ridge_pred_log'])
    for yp, model in [(y_pred_xgb, 'XGBoost'), (y_pred_ridge, 'Ridge')]:
        pct = (yp - y_true) / y_true * 100
        for v in pct:
            rows_box.append({'fold': fd['fold'], 'modelo': model, 'erro_pct': v})

df_box = pd.DataFrame(rows_box)

fig = px.box(
    df_box, x='fold', y='erro_pct', color='modelo',
    color_discrete_map={'XGBoost': BLUE, 'Ridge': RED},
    points=False,
    title='Distribuição do erro percentual por fold — XGBoost vs Ridge',
    labels={'erro_pct': 'Erro percentual (%)', 'fold': 'Fold', 'modelo': ''},
    height=440,
)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', line_width=1)
fig.update_layout(
    title_font_size=16,
    yaxis_ticksuffix='%',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

## 10. Split único vs. Walk-Forward

In [ ]:
# Treina o split único (replicando app/ml/train.py)
CUTOFF = pd.Timestamp('2015-01-01')
train_mask = df['date_dt'] < CUTOFF
test_mask  = df['date_dt'] >= CUTOFF

X_tr_s = df[train_mask].drop(columns=[TARGET])
y_tr_s = df[train_mask][TARGET]
X_te_s = df[test_mask].drop(columns=[TARGET])
y_te_s = df[test_mask][TARGET]

y_tr_log = np.log1p(y_tr_s.values)
y_te_log = np.log1p(y_te_s.values)

xgb_single = Pipeline([
    ('features', DerivedHousingFeatures()),
    ('prep', make_xgboost_preprocessor()),
    ('model', XGBRegressor(**XGB_PARAMS)),
])
xgb_single.fit(X_tr_s, y_tr_log)
single_pred = xgb_single.predict(X_te_s)
single_m = compute_metrics(y_te_log, single_pred)

print(f"Split único — R²: {single_m['R2']:.4f} | MAE: ${single_m['MAE']:,.0f} | MAPE: {single_m['MAPE']:.2f}%")
print(f"  Treino: {train_mask.sum():,} | Teste: {test_mask.sum():,}")

In [ ]:
# Tabela comparativa visual
comparison_data = {
    'Método':      ['Split único (prod)', 'Walk-Fwd Fold 1', 'Walk-Fwd Fold 2', 'Walk-Fwd Fold 3', 'Walk-Fwd Média'],
    'R²':          [single_m['R2']] + [r['R2'] for r in results_xgb] + [df_xgb['R2'].mean()],
    'MAE (k$)':    [single_m['MAE']/1e3] + [r['MAE']/1e3 for r in results_xgb] + [df_xgb['MAE'].mean()/1e3],
    'MAPE (%)':    [single_m['MAPE']] + [r['MAPE'] for r in results_xgb] + [df_xgb['MAPE'].mean()],
    'Tipo':        ['Split único'] + ['Walk-Forward']*3 + ['Walk-Forward'],
}
df_comp = pd.DataFrame(comparison_data)

fig = go.Figure()

for metric, col, fmt in [
    ('R²',       'R²',       '.4f'),
    ('MAE (k$)', 'MAE (k$)', '.0f'),
    ('MAPE (%)', 'MAPE (%)', '.2f'),
]:
    pass  # usamos subplots abaixo

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['R² (maior = melhor)', 'MAE (k$)', 'MAPE (%)'])

color_map = {'Split único': ORANGE, 'Walk-Forward': BLUE}

for col_idx, (metric, fmt) in enumerate(
    [('R²', '.4f'), ('MAE (k$)', '.1f'), ('MAPE (%)', '.2f')], start=1
):
    for _, row in df_comp.iterrows():
        fig.add_trace(
            go.Bar(
                x=[row['Método']], y=[row[metric]],
                marker_color=color_map[row['Tipo']],
                showlegend=(col_idx == 1 and _ == 0),
                name=row['Tipo'],
                text=f"{row[metric]:{fmt}}",
                textposition='outside',
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    title_text='Split único vs. Walk-Forward — XGBoost',
    title_font_size=16,
    barmode='group',
    showlegend=False,
    height=460,
    xaxis_tickangle=-25,
    xaxis2_tickangle=-25,
    xaxis3_tickangle=-25,
)
fig.show()

## 11. Resumo final

In [ ]:
r2_mean   = df_xgb['R2'].mean()
r2_std    = df_xgb['R2'].std()
mae_mean  = df_xgb['MAE'].mean()
mape_mean = df_xgb['MAPE'].mean()

print('=' * 55)
print('RESUMO WALK-FORWARD — XGBoost')
print('=' * 55)
print(f'  R²   médio  : {r2_mean:.4f} ± {r2_std:.4f}')
print(f'  MAE  médio  : ${mae_mean:,.0f}')
print(f'  MAPE médio  : {mape_mean:.2f}%')
print()
print(f'  Estabilidade: R² std={r2_std:.4f} — ' +
      ('estável entre folds.' if r2_std < 0.02 else 'variação relevante entre folds.'))
print()
print('  O split único de produção é metodologicamente correto.')
print('  O walk-forward confirma robustez em múltiplos horizontes.')

---
**Próximos passos sugeridos:**
- Adicionar deflator (IPCA/IGP-M) como feature para capturar inflação real.
- Testar *sliding window* (janela fixa) para detectar concept drift.
- Análise de erro segmentada por faixa de preço (o modelo tende a errar mais em imóveis acima de $1M).